# ShinyySwapper - Professional Face Swap Model Training
## Optimized for Google Colab Free GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch torchvision opencv-python insightface onnxruntime-gpu albumentations kornia lpips facexlib gfpgan scipy kaggle

## Download 80K Training Images (FFHQ + CelebA-HQ)

## Setup Kaggle API (Run Once)

In [ ]:
# Upload your kaggle.json file
from google.colab import files
print('Upload your kaggle.json file (Get it from kaggle.com → Settings → API)')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle API configured!')

In [ ]:
import os
from pathlib import Path
import shutil
from tqdm import tqdm

output_dir = '/content/drive/MyDrive/shinyyswapper_data'
os.makedirs(output_dir, exist_ok=True)

# Install Kaggle
!pip install -q kaggle

# Download FFHQ from Kaggle (70k images)
print('📥 Downloading FFHQ 70k images from Kaggle...')
!kaggle datasets download -d denislukovnikov/ffhq256 -p /content --unzip
!cp -r /content/ffhq256/* {output_dir}/

# Download CelebA-HQ from Kaggle (10k images)
print('📥 Downloading CelebA-HQ 10k images...')
!kaggle datasets download -d badasstechie/celebahq-resized-256x256 -p /content --unzip
!mkdir -p /content/celeba_temp
!cp -r /content/celeba_hq_256/* /content/celeba_temp/ 2>/dev/null || cp -r /content/*.jpg /content/celeba_temp/ 2>/dev/null || true

# Copy first 10k
celeba_imgs = sorted(list(Path('/content/celeba_temp').rglob('*.jpg')))[:10000]
for img in tqdm(celeba_imgs, desc='Copying CelebA'):
    shutil.copy(img, output_dir)

# Cleanup
!rm -rf /content/celeba_temp /content/ffhq256 /content/celeba_hq_256

total = len(list(Path(output_dir).rglob('*.jpg'))) + len(list(Path(output_dir).rglob('*.png')))
print(f'✅ Dataset ready: {total} images')

In [ ]:
# Clone your repo
!git clone https://github.com/ShinyyMineyy/ShinyySwapper.git /content/shinyyswapper
%cd /content/shinyyswapper

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Start training on 80k images (auto-resumes from last checkpoint)
!python train.py

## Export Model for Inference

In [ ]:
import torch
from model import Generator

# Load trained model (80k images, 100 epochs)
G = Generator()
checkpoint = torch.load('/content/drive/MyDrive/shinyyswapper_checkpoints/checkpoint_epoch_99.pt')
G.load_state_dict(checkpoint['G_state'])
G.eval()

# Export to ONNX for fast CPU inference
dummy_img = torch.randn(1, 3, 256, 256)
dummy_id = torch.randn(1, 512)

torch.onnx.export(
    G,
    (dummy_img, dummy_id),
    '/content/drive/MyDrive/shinyyswapper_model.onnx',
    input_names=['target_image', 'source_identity'],
    output_names=['swapped_face'],
    dynamic_axes={'target_image': {0: 'batch'}, 'source_identity': {0: 'batch'}, 'swapped_face': {0: 'batch'}}
)

print("ShinyySwapper model exported successfully!")